# Notebook 8: Auto Router Adapter Prediction

Bu notebook tek goruntu icin tam akisi calistirir: Notebook 1 router yuzeyi crop/part secer, sonra secilen adapter otomatik yuklenir ve hastalik/OOD tahmini uretilir.

Router davranisi Notebook 1'in maintained cell script'lerinden gelir. Notebook 8 sadece router sonucundan sonraki adapter tahmin adimini ekleyen ince wrapper'dir.


## Hizli Akis

- Yeni kullanici icin `NOTEBOOK_PROFILE = 'a100_colab_default'`, `NOTEBOOK_SPEED_MODE = 'fast'`, `IMAGE_PATH = None` yeterlidir.
- Router belirsiz kalirsa adapter zorlanmaz; sonuc `router_uncertain`, `unknown_crop` veya ilgili reddetme status'u olarak kalir.
- Adapter yolu varsayilan olarak config icindeki `models/adapters` kokunden cozulur; gerekiyorsa `ADAPTER_ROOT` verin.
- Bu notebook klasor/batch tahmini yapmaz; tek goruntu yuzeyidir.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell01_bootstrap_access.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Notebook 1 ile ayni router kullanici ayarlari.
NOTEBOOK_PROFILE = 'a100_colab_default'  # custom, a100_colab_default, cpu_debug
NOTEBOOK_SPEED_MODE = 'fast'  # fast, balanced, quality

# Gerekirse son override katmani (opsiyonel).
INFERENCE_OVERRIDES = {}

# Notebook 8 adapter adimi ayarlari.
ADAPTER_ROOT = None
RETURN_OOD = True
PRINT_JSON_RESULT = False

run_cell_script('nb1_cell03_runtime_setup.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Tam otomatik akis: Notebook 1 router analizini calistir, sonra adapteri yukleyip hastalik/OOD tahmini yap.
ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Yeni dosya yolu verin veya None birakip yukleme kullanin.

run_cell_script('nb1_cell04_analysis.py', globals())

router_result = result
run_cell_script('nb8_cell05_adapter_prediction.py', globals())

auto_result
